# 🎬 3D相机多角度图像生成器 (Colab版)
基于 Qwen-Image-Edit-2511 + Multi-Angles LoRA
**完全免费** - 使用 Google Colab 免费 T4 GPU

### 使用步骤:
1. 点击菜单栏 `Runtime` → `Change runtime type` → 选择 `T4 GPU`
2. 按顺序执行每个单元格 (Ctrl+F9 全部执行)
3. 最后会输出一个 Gradio 公开链接，点击即可使用

In [ ]:
# @title 1. 安装依赖 (约5-8分钟)
import subprocess, sys, os, warnings
warnings.filterwarnings("ignore")

print("正在安装依赖...")

# PyTorch CUDA
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Core
!pip install transformers accelerate safetensors sentencepiece peft -q
!pip install huggingface_hub -q

# Diffusers (latest)
!pip install git+https://github.com/huggingface/diffusers.git -q

# GGUF support
!pip install gguf -q

# UI
!pip install gradio -q
!pip install requests pillow numpy -q

print("依赖安装完成!")

# GPU check
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB')
else:
    print('WARNING: No GPU detected!')

In [ ]:
# @title 2. 下载模型 (首次约10分钟, 后续从缓存加载)
from huggingface_hub import hf_hub_download

print("下载 GGUF 量化模型 (7.5GB)...")
gguf_path = hf_hub_download(
    repo_id="unsloth/Qwen-Image-Edit-2511-GGUF",
    filename="qwen-image-edit-2511-Q2_K.gguf"
)
print(f"模型下载完成: {gguf_path}")

In [ ]:
# @title 3. 加载模型 + LoRA
import torch
from diffusers import QwenImageEditPlusPipeline, QwenImageTransformer2DModel, GGUFQuantizationConfig

print("加载 Transformer...")
transformer = QwenImageTransformer2DModel.from_single_file(
    gguf_path,
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
    config="Qwen/Qwen-Image-Edit-2511",
    subfolder="transformer",
)

print("创建 Pipeline...")
pipe = QwenImageEditPlusPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit-2511",
    transformer=transformer,
    torch_dtype=torch.bfloat16,
)

pipe.enable_model_cpu_offload()

print("加载 Lightning LoRA (4步快速推理)...")
pipe.load_lora_weights(
    "lightx2v/Qwen-Image-Edit-2511-Lightning",
    weight_name="Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors",
    adapter_name="lightning"
)

print("加载 Multi-Angles LoRA (相机控制)...")
pipe.load_lora_weights(
    "fal/Qwen-Image-Edit-2511-Multiple-Angles-LoRA",
    weight_name="qwen-image-edit-2511-multiple-angles-lora.safetensors",
    adapter_name="angles"
)

pipe.set_adapters(["lightning", "angles"], adapter_weights=[1.0, 1.0])

print("模型加载完成! ✅")

In [ ]:
# @title 4. 启动 Gradio 应用 (获取公开链接)
import gradio as gr
import numpy as np
import random
from PIL import Image
import io, json, base64

MAX_SEED = np.iinfo(np.int32).max

# === 相机映射 ===
AZIMUTH_MAP = {
    0: "front view", 45: "front-right quarter view",
    90: "right side view", 135: "back-right quarter view",
    180: "back view", 225: "back-left quarter view",
    270: "left side view", 315: "front-left quarter view",
}
ELEVATION_MAP = {
    -30: "low-angle shot", 0: "eye-level shot",
    30: "elevated shot", 60: "high-angle shot",
}
DISTANCE_MAP = {
    0.6: "close-up", 1.0: "medium shot", 1.8: "wide shot",
}

def snap(v, opts):
    return min(opts, key=lambda x: abs(x - v))

def build_prompt(az, el, dist):
    az = snap(az, list(AZIMUTH_MAP.keys()))
    el = snap(el, list(ELEVATION_MAP.keys()))
    dist = snap(dist, list(DISTANCE_MAP.keys()))
    return f"<sks> {AZIMUTH_MAP[az]} {ELEVATION_MAP[el]} {DISTANCE_MAP[dist]}"

# === Three.js 3D相机 HTML ===
def get_3d_html(value=None, image_url=None):
    if value is None:
        value = {"azimuth": 0, "elevation": 0, "distance": 1.0}
    vj = json.dumps(value)
    ij = json.dumps(image_url)
    return f'''<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<div id="c3d-root" style="width:100%;height:420px;position:relative;background:#1a1a2e;border-radius:12px;overflow:hidden;">
<div id="c3d-ov" style="position:absolute;bottom:12px;left:50%;transform:translateX(-50%);background:rgba(0,0,0,0.85);padding:8px 16px;border-radius:8px;font-family:monospace;font-size:13px;color:#0f8;white-space:nowrap;z-index:10;pointer-events:none;"></div>
<div style="position:absolute;top:10px;right:14px;font-size:11px;color:#999;z-index:10;text-align:right;line-height:1.7;pointer-events:none;">🟢方位角<br>🌸仰角<br>🟠距离</div>
</div><script>(()=>{{const root=document.getElementById('c3d-root'),ov=document.getElementById('c3d-ov');let _v={vj},_img={ij};(function init(){{if(typeof THREE==='undefined'){{setTimeout(init,100);return;}}const S=new THREE.Scene();S.background=new THREE.Color(0x1a1a2e);const C=new THREE.PerspectiveCamera(48,root.clientWidth/root.clientHeight,0.1,100);C.position.set(4.5,3.2,4.5);C.lookAt(0,0.7,0);const R=new THREE.WebGLRenderer({{antialias:true}});R.setSize(root.clientWidth,root.clientHeight);R.setPixelRatio(Math.min(devicePixelRatio,2));root.insertBefore(R.domElement,ov);S.add(new THREE.AmbientLight(0xffffff,0.55));const dl=new THREE.DirectionalLight(0xffffff,0.55);dl.position.set(5,10,5);S.add(dl);S.add(new THREE.GridHelper(8,16,0x335,0x224));const CTR=new THREE.Vector3(0,0.75,0),BD=1.6,AR=2.4,ER=1.8;const aSteps=[0,45,90,135,180,225,270,315],eSteps=[-30,0,30,60],dSteps=[0.6,1.0,1.8];const aN={{0:'front view',45:'front-right quarter view',90:'right side view',135:'back-right quarter view',180:'back view',225:'back-left quarter view',270:'left side view',315:'front-left quarter view'}};const eN={{'-30':'low-angle shot','0':'eye-level shot','30':'elevated shot','60':'high-angle shot'}};const dN={{'0.6':'close-up','1.0':'medium shot','1.8':'wide shot'}};const snap=(v,s)=>s.reduce((a,b)=>Math.abs(b-v)<Math.abs(a-v)?b:a);let az=_v.azimuth||0,el=_v.elevation||0,dist=_v.distance||1.0;const pMat=new THREE.MeshBasicMaterial({{map:null,side:THREE.DoubleSide,transparent:true,opacity:0.95}});let tPlane=new THREE.Mesh(new THREE.PlaneGeometry(1.2,1.2),pMat);tPlane.position.copy(CTR);S.add(tPlane);function mkPh(){{const c=document.createElement('canvas');c.width=256;c.height=256;const x=c.getContext('2d');x.fillStyle='#2a2a3e';x.fillRect(0,0,256,256);x.fillStyle='#fc9';x.beginPath();x.arc(128,128,70,0,Math.PI*2);x.fill();x.fillStyle='#333';x.beginPath();x.arc(100,108,9,0,Math.PI*2);x.arc(156,108,9,0,Math.PI*2);x.fill();x.strokeStyle='#333';x.lineWidth=3;x.beginPath();x.arc(128,130,30,0.2,Math.PI-0.2);x.stroke();x.fillStyle='#adf';x.font='14px sans-serif';x.textAlign='center';x.fillText('上传图片',128,195);x.fillText('拖拽手柄选角度',128,218);return new THREE.CanvasTexture(c);}}pMat.map=mkPh();pMat.needsUpdate=true;function updTex(url){{if(!url){{pMat.map=mkPh();pMat.needsUpdate=true;return;}}new THREE.TextureLoader().load(url,tex=>{{tex.minFilter=THREE.LinearFilter;tex.magFilter=THREE.LinearFilter;pMat.map=tex;pMat.needsUpdate=true;const img=tex.image;if(img&&img.width&&img.height){{const asp=img.width/img.height,mx=1.5;let pw=mx,ph=mx/asp;if(asp<1){{ph=mx;pw=mx*asp;}}S.remove(tPlane);tPlane.geometry.dispose();tPlane=new THREE.Mesh(new THREE.PlaneGeometry(pw,ph),pMat);tPlane.position.copy(CTR);S.add(tPlane);}}}},undefined,()=>{{}});}}if(_img)updTex(_img);const camG=new THREE.Group();const bM=new THREE.MeshStandardMaterial({{color:0x69c,metalness:0.5,roughness:0.3}});camG.add(new THREE.Mesh(new THREE.BoxGeometry(0.3,0.22,0.38),bM));const lens=new THREE.Mesh(new THREE.CylinderGeometry(0.09,0.11,0.18,16),new THREE.MeshStandardMaterial({{color:0x69c,metalness:0.5,roughness:0.3}}));lens.rotation.x=Math.PI/2;lens.position.z=0.26;camG.add(lens);S.add(camG);const aRing=new THREE.Mesh(new THREE.TorusGeometry(AR,0.04,16,64),new THREE.MeshStandardMaterial({{color:0x0f8,emissive:0x0f8,emissiveIntensity:0.3}}));aRing.rotation.x=Math.PI/2;aRing.position.y=0.05;S.add(aRing);const aH=new THREE.Mesh(new THREE.SphereGeometry(0.18,16,16),new THREE.MeshStandardMaterial({{color:0x0f8,emissive:0x0f8,emissiveIntensity:0.5}}));aH.userData.type='azimuth';S.add(aH);const arc=[];for(let i=0;i<=32;i++){{const a=THREE.MathUtils.degToRad(-30+90*i/32);arc.push(new THREE.Vector3(-0.8,ER*Math.sin(a)+CTR.y,ER*Math.cos(a)));}}const eArc=new THREE.Mesh(new THREE.TubeGeometry(new THREE.CatmullRomCurve3(arc),32,0.04,8,false),new THREE.MeshStandardMaterial({{color:0xf69,emissive:0xf69,emissiveIntensity:0.3}}));S.add(eArc);const eH=new THREE.Mesh(new THREE.SphereGeometry(0.18,16,16),new THREE.MeshStandardMaterial({{color:0xf69,emissive:0xf69,emissiveIntensity:0.5}}));eH.userData.type='elevation';S.add(eH);const dLG=new THREE.BufferGeometry();const dL=new THREE.Line(dLG,new THREE.LineBasicMaterial({{color:0xfa0}}));S.add(dL);const dH=new THREE.Mesh(new THREE.SphereGeometry(0.18,16,16),new THREE.MeshStandardMaterial({{color:0xfa0,emissive:0xfa0,emissiveIntensity:0.5}}));dH.userData.type='distance';S.add(dH);function upPos(){{const d=BD*dist,aR=THREE.MathUtils.degToRad(az),eR=THREE.MathUtils.degToRad(el);const cx=d*Math.sin(aR)*Math.cos(eR),cy=d*Math.sin(eR)+CTR.y,cz=d*Math.cos(aR)*Math.cos(eR);camG.position.set(cx,cy,cz);camG.lookAt(CTR);aH.position.set(AR*Math.sin(aR),0.05,AR*Math.cos(aR));eH.position.set(-0.8,ER*Math.sin(eR)+CTR.y,ER*Math.cos(eR));const od=d-0.5;dH.position.set(od*Math.sin(aR)*Math.cos(eR),od*Math.sin(eR)+CTR.y,od*Math.cos(aR)*Math.cos(eR));dLG.setFromPoints([camG.position.clone(),CTR.clone()]);const aS=snap(az,aSteps),eS=snap(el,eSteps),dS=snap(dist,dSteps);ov.textContent='<sks> '+aN[aS]+' '+eN[String(eS)]+' '+dN[String(dS)];}}function emit(){{const aS=snap(az,aSteps),eS=snap(el,eSteps),dS=snap(dist,dSteps);window._c3dVal={{azimuth:aS,elevation:eS,distance:dS}};root.dispatchEvent(new CustomEvent('c3d-change',{{detail:window._c3dVal}}));}}const rc=new THREE.Raycaster(),ms=new THREE.Vector2();let drag=false,tgt=null,dsY=0,dsD=1.0;const hp=new THREE.Vector3(),hdls=[aH,eH,dH];function gM(e){{const r=R.domElement.getBoundingClientRect();ms.x=((e.clientX-r.left)/r.width)*2-1;ms.y=-((e.clientY-r.top)/r.height)*2+1;}}function onDn(e){{gM(e.touches?e.touches[0]:e);rc.setFromCamera(ms,C);const h=rc.intersectObjects(hdls);if(h.length>0){{drag=true;tgt=h[0].object;tgt.material.emissiveIntensity=1;tgt.scale.setScalar(1.3);dsY=ms.y;dsD=dist;R.domElement.style.cursor='grabbing';}}}}function onMv(e){{if(e.touches)e.preventDefault();gM(e.touches?e.touches[0]:e);if(drag&&tgt){{rc.setFromCamera(ms,C);const t=tgt.userData.type;if(t==='azimuth'){{const p=new THREE.Plane(new THREE.Vector3(0,1,0),-0.05);if(rc.ray.intersectPlane(p,hp)){{az=THREE.MathUtils.radToDeg(Math.atan2(hp.x,hp.z));if(az<0)az+=360;}}}}else if(t==='elevation'){{const p=new THREE.Plane(new THREE.Vector3(1,0,0),-0.8);if(rc.ray.intersectPlane(p,hp)){{el=THREE.MathUtils.clamp(THREE.MathUtils.radToDeg(Math.atan2(hp.y-CTR.y,hp.z)),-30,60);}}}}else if(t==='distance'){{dist=THREE.MathUtils.clamp(dsD-(ms.y-dsY)*1.5,0.6,1.8);}}upPos();}}else{{rc.setFromCamera(ms,C);const h=rc.intersectObjects(hdls);hdls.forEach(x=>{{x.material.emissiveIntensity=0.5;x.scale.setScalar(1);}});if(h.length>0){{h[0].object.material.emissiveIntensity=0.8;h[0].object.scale.setScalar(1.12);R.domElement.style.cursor='grab';}}else R.domElement.style.cursor='default';}}}function onUp(e){{if(tgt){{tgt.material.emissiveIntensity=0.5;tgt.scale.setScalar(1);const tA=snap(az,aSteps),tE=snap(el,eSteps),tD=snap(dist,dSteps);const sA=az,sE=el,sD=dist,st=Date.now();(function anim(){{const t=Math.min((Date.now()-st)/200,1);const e=1-Math.pow(1-t,3);let d=tA-sA;if(d>180)d-=360;if(d<-180)d+=360;az=sA+d*e;if(az<0)az+=360;if(az>=360)az-=360;el=sE+(tE-sE)*e;dist=sD+(tD-sD)*e;upPos();if(t<1)requestAnimationFrame(anim);else emit();}})();}}drag=false;tgt=null;R.domElement.style.cursor='default';}}R.domElement.addEventListener('mousedown',onDn);R.domElement.addEventListener('mousemove',onMv);R.domElement.addEventListener('mouseup',onUp);R.domElement.addEventListener('mouseleave',onUp);R.domElement.addEventListener('touchstart',onDn,{{passive:false}});R.domElement.addEventListener('touchmove',onMv,{{passive:false}});R.domElement.addEventListener('touchend',onUp,{{passive:false}});R.domElement.addEventListener('touchcancel',onUp,{{passive:false}});upPos();window._c3dVal={{azimuth:snap(az,aSteps),elevation:snap(el,eSteps),distance:snap(dist,dSteps)}};(function render(){{requestAnimationFrame(render);R.render(S,C);}})();new ResizeObserver(()=>{{C.aspect=root.clientWidth/root.clientHeight;C.updateProjectionMatrix();R.setSize(root.clientWidth,root.clientHeight);}}).observe(root);window._c3dSetVal=(v)=>{{if(v){{az=v.azimuth??az;el=v.elevation??el;dist=v.distance??dist;upPos();window._c3dVal={{azimuth:snap(az,aSteps),elevation:snap(el,eSteps),distance:snap(dist,dSteps)}};}}}};window._c3dSetImg=(url)=>{{updTex(url);}};}})();}})();</script>'''

# === 推理函数 ===
def infer(image, azimuth, elevation, distance, seed, randomize_seed, guidance_scale, steps, width, height, progress=None):
    prompt = build_prompt(azimuth, elevation, distance)
    if randomize_seed:
        seed = random.randint(0, MAX_SEED)
    generator = torch.Generator(device="cuda").manual_seed(seed)
    pil_image = image.convert("RGB") if image else None
    if pil_image is None:
        raise gr.Error("请上传图片")
    result = pipe(
        image=[pil_image], prompt=prompt,
        height=height if height != 0 else None,
        width=width if width != 0 else None,
        num_inference_steps=steps, generator=generator,
        guidance_scale=guidance_scale,
        num_images_per_prompt=1,
    ).images[0]
    return result, seed, prompt

# === UI ===
with gr.Blocks(theme=gr.themes.Soft(), title="3D相机多角度图像生成器") as demo:
    gr.Markdown("""
    # 🎬 3D 相机多角度图像生成器 (Colab版)
    上传图片 → 拖拽 3D 手柄选角度 → AI 生成新视角。运行在免费 Colab T4 GPU 上!
    """)

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=400):
            image_input = gr.Image(label="上传图片", type="pil", height=260)
            gr.Markdown("### 3D 相机控制")
            camera_html = gr.HTML(value=get_3d_html())
            camera_state = gr.State({"azimuth": 0, "elevation": 0, "distance": 1.0})
            run_btn = gr.Button("生成", variant="primary", size="lg")
            azimuth_slider = gr.Slider(label="方位角", minimum=0, maximum=315, step=45, value=0)
            elevation_slider = gr.Slider(label="仰角", minimum=-30, maximum=60, step=30, value=0)
            distance_slider = gr.Slider(label="距离", minimum=0.6, maximum=1.8, step=0.4, value=1.0)
            prompt_preview = gr.Textbox(label="提示词", value="<sks> front view eye-level shot medium shot", interactive=False)
        with gr.Column(scale=1, min_width=400):
            image_output = gr.Image(label="生成结果", type="pil", height=400)
            with gr.Accordion("高级设置", open=False):
                seed_input = gr.Number(label="Seed", value=0, precision=0)
                randomize_seed = gr.Checkbox(label="随机Seed", value=True)
                guidance_scale = gr.Slider(label="引导系数", minimum=1.0, maximum=10.0, step=0.5, value=1.0)
                num_steps = gr.Slider(label="推理步数", minimum=1, maximum=20, step=1, value=4)
                width_slider = gr.Slider(label="宽度", minimum=256, maximum=2048, step=8, value=1024)
                height_slider = gr.Slider(label="高度", minimum=256, maximum=2048, step=8, value=1024)

    # 事件
    def update_prompt(az, el, dist):
        return build_prompt(az, el, dist)

    def sync_to_3d(az, el, dist):
        v = {"azimuth": int(az), "elevation": int(el), "distance": float(dist)}
        return v, build_prompt(az, el, dist)

    for s in [azimuth_slider, elevation_slider, distance_slider]:
        s.change(fn=update_prompt, inputs=[azimuth_slider, elevation_slider, distance_slider], outputs=[prompt_preview])
        s.release(fn=sync_to_3d, inputs=[azimuth_slider, elevation_slider, distance_slider], outputs=[camera_state, prompt_preview]
        ).then(fn=None, js="(v)=>{if(window._c3dSetVal)window._c3dSetVal(v);}", inputs=[camera_state], outputs=None)

    run_btn.click(
        fn=infer,
        inputs=[image_input, azimuth_slider, elevation_slider, distance_slider, seed_input, randomize_seed, guidance_scale, num_steps, width_slider, height_slider],
        outputs=[image_output, seed_input, prompt_preview],
    )

    # 图片上传 → 更新3D视口
    def update_3d_img(image):
        if image is None:
            return gr.update(value=get_3d_html())
        buf = io.BytesIO()
        image.convert("RGB").save(buf, format="PNG")
        data_url = f"data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}"
        return gr.update(value=get_3d_html(image_url=data_url))

    image_input.upload(fn=update_3d_img, inputs=[image_input], outputs=[camera_html])
    image_input.clear(fn=lambda: gr.update(value=get_3d_html()), outputs=[camera_html])

print("启动中... 获取公开链接:")
demo.queue().launch(share=True, debug=False, show_error=True)
print("\n上方链接即为公开访问地址，点击即可使用!")